Architecture

Input: (B, 1, 28, 28)

Block 1:
  Conv2d:    1 → 16 channels, kernel 3×3, padding 1
  ReLU
  MaxPool2d: 2×2, stride 2
  → Output: (B, 16, 14, 14)

Block 2:
  Conv2d:    16 → 32 channels, kernel 3×3, padding 1
  ReLU
  MaxPool2d: 2×2, stride 2
  → Output: (B, 32, 7, 7)

Head:
  Flatten (keep batch dim)
  Linear: 32*7*7 → 10
  → Output: (B, 10) logits

Hyperparameters 
Loss: CrossEntropyLoss 
Optimizer: Adam 
LR: 1e-3 
Batch Size: 64 
Epochs: 5 


In [48]:
import torch 
import torch.nn as nn 
import torch.optim as optim 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms 

In [49]:
transform = transforms.Compose([
    transforms.ToTensor(), 
]) 
train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)


In [50]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True) 
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

images, labels = next(iter(train_loader)) 
print(f"Batch shape: {images.shape}")  # (64, 1, 28, 28)
print(f"Labels shape: {labels.shape}")  # (64,)
print(f"Label range: {labels.min()} to {labels.max()}")  # 0 to 9 

Batch shape: torch.Size([64, 1, 28, 28])
Labels shape: torch.Size([64])
Label range: 0 to 9


In [64]:
class CNN(nn.Module): 
    def __init__(self, batch_norm: bool = False): 
        super().__init__()
        self.flatten = nn.Flatten() 
        self.cnn1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1) 
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) 
        self.cnn2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1) 
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) 
        self.linear = nn.Linear(32 * 7 * 7, 10) 
        self.batch_norm = batch_norm 
        self.batch_norm1 = nn.BatchNorm2d(num_features=16)
        self.batch_norm2 = nn.BatchNorm2d(num_features=32)

    def forward(self, x): 
        # first convolutional layer 
        x = self.cnn1(x) 
        if self.batch_norm: 
            x = self.batch_norm1(x)
        x = torch.relu(x) 
        x = self.pool1(x) 
        # print(f"Shape after first layer ", x.shape) 
        # second convolutional layer 
        x = self.cnn2(x)
        if self.batch_norm: 
            x = self.batch_norm2(x)
        x = torch.relu(x) 
        x = self.pool2(x) 
        # print(f"Shape after second layer ", x.shape)
        x = self.flatten(x) 
        x = self.linear(x)
        return x 




In [56]:
from datetime import datetime 
start = datetime.now() 
model = CNN() 
loss_fn = nn.CrossEntropyLoss() 
optimizer = optim.Adam(model.parameters(), lr=1e-3) 
n_epochs = 5
for epoch in range(n_epochs): 
    model.train() 
    running_loss = 0.0 
    correct = 0 
    total = 0 
    for images, labels in train_loader: 
        optimizer.zero_grad() 
        logits = model(images) 
        loss = loss_fn(logits, labels)
        loss.backward() 
        optimizer.step() 

        running_loss += loss.item() 
        preds = logits.argmax(dim=1) 
        correct += (preds == labels).sum().item() 
        total += labels.size(0)
    train_acc = correct / total 
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}: loss={avg_loss:.4f}, train_acc={train_acc:.4f}") 
end = datetime.now() 
training_finished_without_batch_norm = (end - start).total_seconds() 
print(f"Total Training Time: {training_finished_without_batch_norm} seconds")

Epoch 1: loss=0.5203, train_acc=0.8139
Epoch 2: loss=0.3472, train_acc=0.8776
Epoch 3: loss=0.3077, train_acc=0.8911
Epoch 4: loss=0.2825, train_acc=0.9001
Epoch 5: loss=0.2625, train_acc=0.9066
Total Training Time: 71.058665 seconds


In [57]:
model.eval() 
correct = 0 
total = 0
with torch.no_grad():  
    for images, labels in test_loader: 
        logits = model(images) 
        preds = logits.argmax(dim=1) 
        correct += (preds == labels).sum().item() 
        total += labels.size(0)
    print(f"Test Accuracy: {correct / total}:.4f")

Test Accuracy: 0.8871:.4f


In [66]:
from datetime import datetime 
start = datetime.now() 
model2 = CNN(batch_norm=True) 
loss_fn = nn.CrossEntropyLoss() 
optimizer = optim.Adam(model2.parameters(), lr=1e-3) 
n_epochs = 5
for epoch in range(n_epochs): 
    model2.train() 
    running_loss = 0.0 
    correct = 0 
    total = 0 
    for images, labels in train_loader: 
        optimizer.zero_grad() 
        logits = model2(images) 
        loss = loss_fn(logits, labels)
        loss.backward() 
        optimizer.step() 

        running_loss += loss.item() 
        preds = logits.argmax(dim=1) 
        correct += (preds == labels).sum().item() 
        total += labels.size(0)
    train_acc = correct / total 
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}: loss={avg_loss:.4f}, train_acc={train_acc:.4f}") 
end = datetime.now() 
training_finished_with_batch_norm = (end - start).total_seconds() 
print(f"Total Training Time: {training_finished_with_batch_norm} seconds")

Epoch 1: loss=0.3987, train_acc=0.8591
Epoch 2: loss=0.2878, train_acc=0.8965
Epoch 3: loss=0.2546, train_acc=0.9094
Epoch 4: loss=0.2359, train_acc=0.9161
Epoch 5: loss=0.2175, train_acc=0.9216
Total Training Time: 83.950674 seconds


In [67]:
model2.eval() 
correct = 0 
total = 0
with torch.no_grad():  
    for images, labels in test_loader: 
        logits = model2(images) 
        preds = logits.argmax(dim=1) 
        correct += (preds == labels).sum().item() 
        total += labels.size(0)
    print(f"Test Accuracy: {correct / total}:.4f")

Test Accuracy: 0.8967:.4f
